[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/toma-decisiones-mcda/blob/main/04_electre_cacao.ipynb)

# ELECTRE, caso cacao

Mismo caso. Contenido completo en la Sesión 4 (ELECTRE/PROMETHEE) del curso.

**Por qué es una implementación manual y no `pyDecision.algorithm.electre_i`:** la discordancia de la función empaquetada usa una normalización distinta a la fórmula enseñada en clase (S4 §1.3), sus concordancias sí coinciden, las discordancias no exactamente. Para que este notebook reproduzca EXACTAMENTE los números de las diapositivas, se implementa la fórmula directamente.

In [1]:
import itertools

zonas = ["Bonda", "Guachaca", "San Pedro", "Palmor"]
datos = {
    "Bonda":     [27, 78, 6.2, 0.9],
    "Guachaca":  [26, 85, 5.8, 1.5],
    "San Pedro": [24, 82, 6.5, 0.7],
    "Palmor":    [22, 88, 5.5, 1.2],
}
pesos = [0.25, 0.30, 0.25, 0.20]
es_beneficio = [True, False, True, False]  # Temp, Humedad, pH, CE

## Concordancia y discordancia

In [2]:
rangos = [max(datos[z][j] for z in zonas) - min(datos[z][j] for z in zonas)
          for j in range(4)]

def mejor_o_igual(a, b, j):
    return a >= b if es_beneficio[j] else a <= b

def estrictamente_mejor(a, b, j):
    return a > b if es_beneficio[j] else a < b

concordancia, discordancia = {}, {}
for a, b in itertools.permutations(zonas, 2):
    c = sum(w for j, w in enumerate(pesos)
            if mejor_o_igual(datos[a][j], datos[b][j], j))
    d = max([abs(datos[b][j] - datos[a][j]) / rangos[j]
             for j in range(4)
             if estrictamente_mejor(datos[b][j], datos[a][j], j)], default=0.0)
    concordancia[(a, b)] = c
    discordancia[(a, b)] = d

print("Concordancia:", {k: round(v, 2) for k, v in concordancia.items()})
print("Discordancia:", {k: round(v, 2) for k, v in discordancia.items()})

Concordancia: {('Bonda', 'Guachaca'): 1.0, ('Bonda', 'San Pedro'): 0.55, ('Bonda', 'Palmor'): 1.0, ('Guachaca', 'Bonda'): 0, ('Guachaca', 'San Pedro'): 0.25, ('Guachaca', 'Palmor'): 0.8, ('San Pedro', 'Bonda'): 0.45, ('San Pedro', 'Guachaca'): 0.75, ('San Pedro', 'Palmor'): 1.0, ('Palmor', 'Bonda'): 0, ('Palmor', 'Guachaca'): 0.2, ('Palmor', 'San Pedro'): 0}
Discordancia: {('Bonda', 'Guachaca'): 0.0, ('Bonda', 'San Pedro'): 0.3, ('Bonda', 'Palmor'): 0.0, ('Guachaca', 'Bonda'): 0.75, ('Guachaca', 'San Pedro'): 1.0, ('Guachaca', 'Palmor'): 0.38, ('San Pedro', 'Bonda'): 0.6, ('San Pedro', 'Guachaca'): 0.4, ('San Pedro', 'Palmor'): 0.0, ('Palmor', 'Bonda'): 1.0, ('Palmor', 'Guachaca'): 0.8, ('Palmor', 'San Pedro'): 1.0}


## Relación de superación (c* = 0.65, d* = 0.30, convención del curso, S4 §1.4)

In [3]:
c_estrella, d_estrella = 0.65, 0.30
print("a supera a b:")
for a, b in itertools.permutations(zonas, 2):
    if concordancia[(a, b)] >= c_estrella and discordancia[(a, b)] <= d_estrella:
        print(f"  {a} supera a {b}  (c={concordancia[(a,b)]:.2f}, d={discordancia[(a,b)]:.2f})")

a supera a b:
  Bonda supera a Guachaca  (c=1.00, d=0.00)
  Bonda supera a Palmor  (c=1.00, d=0.00)
  San Pedro supera a Palmor  (c=1.00, d=0.00)


**Resultado esperado** (coincide con lo publicado en la Sesión 4 del curso): Bonda supera a Guachaca, Bonda supera a Palmor, San Pedro supera a Palmor, **Bonda y San Pedro quedan incomparables entre sí** (ninguna relación cumple el umbral de concordancia).